# Here is some UX Designer that will help you suggesting some improvements

In [ ]:
import os
import logging
from contextlib import contextmanager
from dotenv import load_dotenv

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

from openai import OpenAI

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
MAX_TEXT_LENGTH = 12000
SCROLLS = 5

# ---------------------------------------------------------
# Environment
# ---------------------------------------------------------
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY missing")

client = OpenAI()

# ---------------------------------------------------------
# Logging
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.WARNING,
    format="[%(levelname)s] %(message)s"
)
logging.disable(logging.WARNING)

# ---------------------------------------------------------
# Selenium driver manager
# ---------------------------------------------------------
def create_driver():
    options = Options()

    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")

    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )


@contextmanager
def browser():
    driver = create_driver()
    try:
        yield driver
    finally:
        driver.quit()

# ---------------------------------------------------------
# Website scraper
# ---------------------------------------------------------
class Website:

    def __init__(self, url):
        self.url = url
        self.title = ""
        self.text = ""    

    def load(self):

        with browser() as driver:

            logging.info(f"🌐 Loading {self.url}")

            driver.get(self.url)

            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            self._scroll_page(driver)

            html = driver.page_source
            self.title = driver.title

        self.text = self._extract_text(html)

    def _scroll_page(self, driver):

        last_height = driver.execute_script(
            "return document.body.scrollHeight"
        )

        for _ in range(SCROLLS):

            driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight);"
            )

            WebDriverWait(driver, 5).until(
                lambda d: d.execute_script(
                    "return document.body.scrollHeight"
                ) >= last_height
            )

            new_height = driver.execute_script(
                "return document.body.scrollHeight"
            )

            if new_height == last_height:
                break

            last_height = new_height

    def _extract_text(self, html):

        soup = BeautifulSoup(html, "html.parser")

        remove_tags = [
            "script", "style", "noscript",
            "svg", "img", "meta",
            "header", "footer", "nav",
            "aside", "form"
        ]

        for tag in soup(remove_tags):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)

        lines = [line.strip() for line in text.splitlines() if line.strip()]

        clean = "\n".join(lines)

        return clean[:MAX_TEXT_LENGTH]

# ---------------------------------------------------------
# OpenAI prompt
# ---------------------------------------------------------
def build_messages(site):

    return [
        {
            "role": "system",
            "content": (
                "You are a professional UX designer."
                "Suggest 2 concrete UX improvements."
                "Keep it short."
                "Each improvement needs to have Problem/Solution/Reason"
                "Format it as Markup"
            ),
        },
        {
            "role": "user",
            "content": f"""
Website title: {site.title}

Content:
{site.text}
""",
        },
    ]

# ---------------------------------------------------------
# AI analysis
# ---------------------------------------------------------
def analyze(url):

    site = Website(url)
    site.load()

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=build_messages(site),
    )

    return response.choices[0].message.content

def get_ux_summary(url): 
    summary = analyze(url) 
    display(Markdown(summary))

get_ux_summary("https://yellowhammer.ovh")
